# 1. Introduction

## 1.1. Glaucoma Detection with a Deep Learning Model Using Fundus Images

Glaucoma is a group of eye conditions that damage the optic nerve, which is crucial for maintaining good vision. This damage is often a result of abnormally high pressure in the eye. It is one of the leading causes of blindness in individuals over the age of 60.

---
In this project by AirTon, we aim to leverage the power of Deep Learning to detect Glaucoma using fundus images. Fundus images are photographs of the interior surface of the eye and include structures such as the retina, optic disc, macula, and posterior pole (back of the eye). These images provide a wealth of information and are commonly used in diagnosing various eye diseases, including Glaucoma.

# 2. Libraries

In [ ]:
! pip install monai==1.3.0

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import os
from glob import glob
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from scipy import ndimage
import cv2

from typing import List, Tuple, Dict, Union, Callable, Iterable

In [ ]:
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:

# Download and unzip the dataset (update dataset-id and zip file name)
!kaggle datasets download -d sshikamaru/glaucoma-detection
!unzip -q glaucoma-detection.zip -d /content/dataset

## 2.1. Deterministic Code

In [ ]:
import random
import numpy as np
import torch
from monai.utils import set_determinism

def make_deterministic(seed):
    """
    Make the code deterministic using the specified seed for Python, PyTorch, and MONAI.

    Args:
        seed (int): The seed to use for random number generation.
    """
    # The seed for Python's random module
    random.seed(seed)

    # The seed for numpy
    np.random.seed(seed)

    # The seed for PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # The seed for MONAI
    set_determinism(seed=seed)

def deterministic(func):
    """
    A decorator that makes a function deterministic.
    """
    def wrapper(*args, **kwargs):
        if 'random_seed' in kwargs:
            random_seed = kwargs['random_seed']
        else:
            random_seed = 0
        make_deterministic(seed=random_seed)
        return func(*args, **kwargs)
    return wrapper

In [ ]:
# Selecting the processor device. Making sure our accelerator is set to GPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device is: {device}")

# 3. Dataset

In [ ]:
meta = pd.read_csv("/content/dataset/glaucoma.csv")
meta.head()

In [ ]:
meta.info()

In [ ]:
# Distribution of diagnosis in our dataset: 0: Normal, 1: Glaucoma
meta["Glaucoma"].value_counts()

In [ ]:
img_root = "/content/dataset/ORIGA/ORIGA/Images"
mask_root = "/content/dataset/ORIGA/ORIGA/Semi-automatic-annotations"

# Adding image and mask roots to the dataframe
meta["Image Path"] = meta["Filename"].apply(lambda fn: os.path.join(img_root, fn))
meta["Mask Path"] = meta["Filename"].apply(lambda fn: os.path.join(mask_root, f"{fn.split('.')[0]}.mat"))
meta.head()

### 3.1. Exploring Image Data

In [ ]:
def remove_black_padding(img, mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnt = max(contours, key=cv2.contourArea)
    x,y,w,h = cv2.boundingRect(cnt)

    p = 30
    crop = img[y-p:y+h+p, x-p:x+w+p]
    crop = cv2.resize(crop, (400, 300))
    return crop

In [ ]:
def get_subject_images(row):
    original_image = cv2.imread(row["Image Path"])

    # Loading mask from .mat file and extract numpy array of the mask, then make it one class
    mat = scipy.io.loadmat(row["Mask Path"])
    binary_mask = (mat['mask'] > 0).astype(np.uint8)
    mask = cv2.cvtColor(binary_mask*255, cv2.COLOR_GRAY2RGB)

    # Apply mask to the original image
    if mask.size != original_image.size:
        shape = original_image.shape
        mask = cv2.resize(mask, (shape[1], shape[0]))

    masked = original_image * (mask > 0)

    # Remove Black Padding
    removed_padding = remove_black_padding(original_image, binary_mask)

    all_images = {
        "Original Image": original_image,
        "Mask": mask,
        "Masked Image": masked,
        "Removed Padding": removed_padding
    }

    return all_images

In [ ]:
import scipy.io

@deterministic
def visualize_images(num_subjects=3):
    num_cols = 4
    fig, axs = plt.subplots(num_subjects, num_cols, figsize=(num_cols*3, num_cols*num_subjects))

    for i, row in meta.sample(n=num_subjects).reset_index().iterrows():
        all_images = get_subject_images(row)
        for j, (title, image) in enumerate(all_images.items()):
            axs[i, j].imshow(image[..., ::-1])
            axs[i, j].set_title(title)

    plt.tight_layout()
    plt.show()

visualize_images(num_subjects=3)

### 3.2. Save Masked and Cropped Images

In [ ]:
from tqdm.notebook import tqdm

save_root = "/content/working/masked-images"
os.makedirs(save_root, exist_ok=True)

for i, row in tqdm(meta.iterrows(), total=len(meta)):
    all_images = get_subject_images(row)

    new_root = os.path.join(save_root, f"{row['Filename']}")
    cv2.imwrite(new_root, all_images["Removed Padding"])
    meta.at[i, "Cropped Path"] = new_root

### Building datasets and data loaders

In [ ]:
from monai.transforms import (LoadImageD, EnsureChannelFirstD, ResizeD, Compose,
                              NormalizeIntensityD, RandRotateD, RandZoomD,
                              LambdaD, ToTensorD, RepeatChannelD,
                              Rotate90d, SelectItemsd)
from monai.data import Dataset
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

In [ ]:
class GlaucomaDataset(Dataset):
    """
    A PyTorch Dataset class for the Glaucoma dataset.
    """

    def __init__(self, df, transform):
        """
        Initialize the dataset.

        Args:
            df (pandas.DataFrame): The dataframe containing the data.
            transform (callable): The transformations to apply to the data.
        """
        self.df = df
        self.transform = transform
        self.data = self.get_data()

    def get_data(self):
        """
        Extract the data from the dataframe.

        Returns:
            list: A list of tuples where each tuple contains the path to an image and its corresponding label.
        """
        data = []
        for i, row in self.df.iterrows():
            label = row["Glaucoma"]
            path = row["Cropped Path"]
            data.append((path, label))
        return data

    def __len__(self):
        """
        Get the number of samples in the dataset.

        Returns:
            int: The number of samples in the dataset.
        """
        return len(self.data)

    def __getitem__(self, idx):
        """
        Get a sample from the dataset and apply monai tranformations to it.

        Args:
            idx (int): The index of the sample to get.

        Returns:
            dict: A dictionary containing the image and its corresponding label.
        """
        image_path, label = self.data[idx]
        item_dict = {"image": image_path, "label": label}
        return self.transform(item_dict)

## 3.3 Glaucoma Dataset using `monai.transforms`
#### The transforms are applied to the fundus images for both training and testing datasets.


In [ ]:
import monai
import numpy as np
from monai.transforms import (LoadImageD, EnsureChannelFirstD, ResizeD, Compose,
                            NormalizeIntensityD, RandRotateD, RandZoomD,
                            ToTensorD, SelectItemsd)

# Monkey patch the problematic set_random_state function
def patched_set_random_state(self, seed=None, state=None):
    if state is not None:
        self.R = state
        return state
    if seed is not None:
        if not isinstance(seed, (int, np.integer)):
            # Use a smaller value instead of id(seed)
            _seed = 123  # Fixed safe value
        else:
            _seed = seed % (2**32 - 1)  # Make sure it fits in uint32
        self.R = np.random.RandomState(_seed)
        return self.R
    self.R = np.random.RandomState()
    return self.R

# Apply the monkey patch
from monai.transforms.transform import RandomizableTransform
RandomizableTransform.set_random_state = patched_set_random_state

# Set determinism globally with a safe seed
monai.utils.set_determinism(seed=123)

image_size = 224

# Building MONAI transforms
aug_transforms = Compose([
    LoadImageD(keys="image"),
    EnsureChannelFirstD(keys="image"),
    ResizeD(keys="image", spatial_size=(image_size, image_size)),
    NormalizeIntensityD(keys="image"),
    RandRotateD(keys="image", mode="bilinear", range_x=0.26, prob=0.5),
    RandZoomD(keys="image", mode="bilinear"),
    ToTensorD(keys=["image", "label"]),
    SelectItemsd(keys=["image", "label"])
])

no_aug_transforms = Compose([
    LoadImageD(keys="image"),
    EnsureChannelFirstD(keys="image"),
    ResizeD(keys="image", spatial_size=(image_size, image_size)),
    NormalizeIntensityD(keys="image"),
    ToTensorD(keys=["image", "label"]),
    SelectItemsd(keys=["image", "label"])
])

In [ ]:
@deterministic
def build_dataloaders():
    meta_nl = meta.loc[meta["Glaucoma"] == 0]
    meta_gl = meta.loc[meta["Glaucoma"] == 1]
    new_meta = pd.concat([meta_nl.sample(n=len(meta_gl)), meta_gl]).copy()

    train_df, test_df = train_test_split(new_meta, test_size=0.25)

    train_data = GlaucomaDataset(train_df, aug_transforms)
    test_data = GlaucomaDataset(test_df, no_aug_transforms)

    idx = 0
    sample = train_data[idx]
    print(f"Image Shape: {sample['image'].shape} | Label: {sample['label']}\n")

    bs = 8
    train_dl = DataLoader(train_data, batch_size=bs, shuffle=True, drop_last=True)
    test_dl = DataLoader(test_data, batch_size=bs, shuffle=False)

    return train_dl, test_dl

train_dl, test_dl = build_dataloaders()

sample_batch = next(iter((train_dl)))
print(f"batch['image'] shape: {sample_batch['image'].shape}")
print(f"batch['image'] dtype: {sample_batch['image'].dtype}")
print(f"batch['label'] shape: {sample_batch['label'].shape}")
print(f"batch['label'] dtype: {sample_batch['label'].dtype}")

In [ ]:
# Plotting sample images from the dataloaders.

@deterministic
def plot_sample_dataloader_images(dataloader):
    """
    Plot 9 sample images from either the training or test dataloaders.
    """
    label_tensor_dict = {1:'GLAUCOMA', 0:'NORMAL'}
    dataiter = iter(dataloader)
    image_tensor_list = list()
    label_list = list()

    fig, axes = plt.subplots(2, 4, figsize=(12, 6))

    data = next(dataiter)
    image_batch, label_batch = data['image'], data['label']
    for i in range(8):
        axes[i//4, i%4].imshow(image_batch[i].permute(1, 2, 0))
        axes[i//4, i%4].set_title(label_tensor_dict[int(label_batch[i])])
    plt.show()

plot_sample_dataloader_images(train_dl)

# 4. Deep Learning Model

In [ ]:
# Import necessary libraries
import torchvision.models as models
import torch.nn as nn
import shutil

@deterministic
def build_model(arch = 'vgg16', pretrained = False):
    """
    Build a resnet model using Pytorch.
    Parameters:
        - arch (str): baseline architecture of the model that could be called using
        torchvision.models.arch command.
        - pretrained (bool): whether or not to use pretrained weights.
    """

    # Loading a model with the specified architecture from torch

    if 'vgg' not in arch and 'alexnet' not in arch and 'resnet' not in arch:
        raise ValueError('Only resnet, vgg or alexnet models can be loaded!')
    else:
        try:
            model = eval(f'models.{arch}(pretrained={pretrained})')
        except:
            raise ValueError('The name of the architecture is not valid!')

    # Replacing the final fully connected layer of the model
    # We have two classes, and therefore, the final FC layer will have two final nodes
    # The VGG network has no FC layer, so we directly change its final layer
    if 'vgg' in arch or 'alexnet' in arch:
        model.classifier._modules['6'] = nn.Linear(4096, 2)
    else:
        num_in_features = model.fc.in_features
        model.fc = nn.Linear(num_in_features, 2)

    return model

In [ ]:
! pip install torchsummary

In [ ]:
from torchsummary import summary

# Plotting the structure for a Vgg16 model
non_trained_vgg16 = build_model(arch='vgg16', pretrained=False).to(device)
summary(non_trained_vgg16, input_size=(3, 224, 224))

# 5. Training


In [ ]:
def train_classifier(model: torch.nn.Module,
                     model_name: str,
                     train_loader: Iterable,
                     valid_loader: Iterable,
                     criterion: Callable,
                     optimizer: torch.optim,
                     num_epochs: int,
                     plot_curves: bool):
    """
    Train a classifier model using pytorch and the given parameters.
    Parameters:
    model (Callable): any pytorch module,
    model_name (str): name of the model to be saved.
    train_loader (Iterable): dataloader for training data,
    valid_loader (Iterable): dataloader for validation data,
    criterion (Callable): any loss function,
    optimizer (torch.optim): a pytorch optimizer,
    num_epochs (int): number of epochs to train the model,
    plot_charts (bool): whether or not to plot the training and validation loss
    and accuracy curves.
    """

    # Sending the model to device (preferably GPU)
    model.to(device)

    # Releasing the GPU memory. This is not necessary, but a good practice to do
    # before starting new training sessions.
    with torch.no_grad():
        torch.cuda.empty_cache()

    # Building a saving directory for models
    model_save_dir = os.path.join('Best_Models', model_name)
    if os.path.exists(model_save_dir):
        shutil.rmtree(model_save_dir)

    os.makedirs(model_save_dir, exist_ok=True)

    # lists to log the epoch values
    epoch_train_loss_list = list()
    epoch_train_accuracy_list = list()
    epoch_valid_loss_list = list()
    epoch_valid_accuracy_list = list()

    # Starting the training loop
    for epoch in range(1, num_epochs+1):
        print("-" * 20, f'\nStarting epoch: {epoch}/{num_epochs}')

        ## training
        model.train()
        steps_train_loss = 0.0
        steps_correct_predictions = 0

        for batch in tqdm(train_loader, unit="batch"):
            inputs, labels = batch['image'].to(device), batch['label'].to(device)

            ### Zero the parameter gradients
            optimizer.zero_grad()

            ### Forward + backward + optimize
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            ### Accumulating the loss and number of correct predictions in step
            steps_train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            steps_correct_predictions += (predicted == labels).sum().item()

        ## Measuring the epoch training loss and accuracy
        epoch_train_loss = steps_train_loss/ len(train_loader)
        epoch_train_loss_list.append(epoch_train_loss)
        epoch_train_accuracy = steps_correct_predictions / len(train_loader.dataset)
        epoch_train_accuracy_list.append(epoch_train_accuracy)

        ## validation
        model.eval()
        steps_valid_loss = 0.0
        steps_correct_predictions = 0

        for batch in tqdm(valid_loader, unit="batch"):
            inputs, labels = batch['image'].to(device), batch['label'].to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            ### Accumulating the loss and number of correct predictions in step
            steps_valid_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            steps_correct_predictions += (predicted == labels).sum().item()

        ## Measuring the epoch validation loss and accuracy
        epoch_valid_loss = steps_valid_loss/ len(valid_loader)
        epoch_valid_loss_list.append(epoch_valid_loss)
        epoch_valid_accuracy = steps_correct_predictions / len(valid_loader.dataset)
        epoch_valid_accuracy_list.append(epoch_valid_accuracy)

        ## Printing the logs
        print(f'train loss: {epoch_train_loss:.2f} | \
        train accuracy: {epoch_train_accuracy:.2f}')
        print(f'valid loss: {epoch_valid_loss:.2f} | \
        valid accuracy: {epoch_valid_accuracy:.2f}')

        ## Saving the best model
        if epoch==1:
            best_valid_loss = epoch_valid_loss
            weight_name = f'{model_name}_Epoch{epoch}_ACC={epoch_valid_accuracy}.pth'
            torch.save(model.state_dict(), os.path.join(model_save_dir, weight_name))
        elif epoch_valid_loss < best_valid_loss:
            weight_name = f'{model_name}_Epoch{epoch}_ACC={epoch_valid_accuracy}.pth'
            torch.save(model.state_dict(), os.path.join(model_save_dir, weight_name))
            best_valid_loss = epoch_valid_loss

    # Printing the best model
    print(f'\nTraining was over. The best model was: {weight_name}')

    # Plotting the loss and accuracy curves
    if plot_curves:
        epoch_train_loss_list.insert(0, 0)
        epoch_train_accuracy_list.insert(0, 0)
        epoch_valid_loss_list.insert(0, 0)
        epoch_valid_accuracy_list.insert(0, 0)
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].plot(epoch_train_loss_list, color='blue', label='training')
        axes[0].plot(epoch_valid_loss_list, color='orange', label='validation')
        axes[0].set_xlim(1, len(epoch_valid_loss_list)+1)
        axes[0].set_title('Loss curves')
        axes[0].legend()
        axes[1].plot(epoch_train_accuracy_list, color='blue', label='training')
        axes[1].plot(epoch_valid_accuracy_list, color='orange', label='validation')
        axes[1].set_xlim(1, len(epoch_train_accuracy_list)+1)
        axes[1].set_title('Accuracy curves')
        axes[1].legend();

    # Loading the best weights and returning the model
    model.load_state_dict(torch.load(os.path.join(model_save_dir, weight_name)))
    return model

In [ ]:
vgg19_model_1 = build_model(arch='vgg19', pretrained=True)
criterion = torch.nn.CrossEntropyLoss()
learning_rate = 0.001
optimizer = torch.optim.SGD(vgg19_model_1.parameters(), lr=learning_rate)
num_epochs = 30
vgg19_model_1 = train_classifier(model=vgg19_model_1,
                                 model_name='vgg19_model_1',
                                 train_loader=train_dl,
                                 valid_loader=test_dl,
                                 criterion=criterion,
                                 optimizer=optimizer,
                                 num_epochs=num_epochs,
                                 plot_curves=True)

# 6. Evaluation

## 6.1. Concepts
 <br><img src="https://i.ibb.co/JCPxvzJ/img5.jpg" alt="img5" border="0" height="100"><u><br>

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

@deterministic
def evaluate_model(model: Callable,
                   test_loader: Iterable = test_dl,
                   plot_cm: bool = True) -> float:
    """
    Evaluate a given model's performance on the test set.
    Parameters:
        model (Callable): the pytorch model to be evaluated.
        test_loader (Iterable): test dataloader to be used as the test data.
        plot_cm (bool): whether or not to plot a confusion matrix.
        accuracy (float): accuracy of the model.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    labels_list = list()
    preds_list = list()
    with torch.no_grad():
        for batch in tqdm(test_dl):
            inputs, labels = batch['image'].to(device), batch['label'].to(device)
            outputs = model(inputs)
            preds = torch.argmax(torch.softmax(outputs, dim=1), dim=1)
            labels_list.append(labels)
            preds_list.append(preds)
        all_labels = torch.cat(labels_list).cpu()
        all_preds = torch.cat(preds_list).cpu()
        accuracy = (all_preds == all_labels).sum().item()/len(all_labels)

    if plot_cm:
        cm = confusion_matrix(all_labels.numpy(), all_preds.numpy())
        ax = plt.subplot()
        sns.heatmap(cm, annot=True, fmt='g', ax=ax)
        ax.set_xlabel('Predicted labels')
        ax.set_ylabel('True labels')
        ax.set_title(f'Confusion Matrix - accuracy: {accuracy:.2f}')
        ax.xaxis.set_ticklabels(['Glaucoma', 'Normal'])
        ax.yaxis.set_ticklabels(['Glaucoma', 'Normal'])

    return accuracy

In [ ]:
evaluate_model(vgg19_model_1)

In [ ]:
# Defining a function to freeze the resnet models for fine-tunning

def freeze_resnet18(resnet18_model: Callable,
                    children_num_to_freeze: int = 5,
                    print_children: bool = False):
    """
    Freeze a resnet18 model up to a user-specified children module.
    Parameters:
    resnet18_model: The resnet18 model to be frozen and returned.
    children_num_to_freeze: The number of children modules to be frozen.
    print_children (bool): Whether or not to print the children modules.
    """
    for i, child in enumerate(resnet18_model.children()):
        if i < children_num_to_freeze:
            status = 'Frozen'
            for param in child.parameters():
                param.requires_grad = False
        else:
            status = 'Unfrozen'
        for param in child.parameters():
            param.requires_grad = True

        if print_children:
            print(f'******************* child module number: {i} - {status}')
            print(child)
    return resnet18_model

In [ ]:
resnet18_model = build_model(arch='resnet18', pretrained=True)
resnet18_model = freeze_resnet18(resnet18_model,
                                   children_num_to_freeze = 5,
                                   print_children=False)


criterion = torch.nn.CrossEntropyLoss()
learning_rate = 0.01
optimizer = torch.optim.SGD(resnet18_model.parameters(), lr=learning_rate,
                            weight_decay=0.001)
num_epochs = 30
resnet18_model_2 = train_classifier(model=resnet18_model,
                                 model_name='resnet18_model_2',
                                 train_loader=train_dl,
                                 valid_loader=test_dl,
                                 criterion=criterion,
                                 optimizer=optimizer,
                                 num_epochs=num_epochs,
                                 plot_curves=True)

In [ ]:
evaluate_model(resnet18_model_2)

# Experimentation...

In [ ]:
# Import necessary libraries
import torchvision.models as models
import torch.nn as nn

def build_model(arch='densenet169', pretrained=False):
    """
    Build a CNN model using PyTorch with custom dense layers.
    Parameters:
        - arch (str): Model architecture to use from torchvision.models.
        - pretrained (bool): Whether to use pretrained weights.
    """

    # Load specified architecture from torchvision models
    if 'densenet' not in arch and 'vgg' not in arch and 'alexnet' not in arch and 'resnet' not in arch:
        raise ValueError('Only densenet, resnet, vgg, or alexnet models can be loaded!')
    else:
        try:
            model = eval(f'models.{arch}(pretrained={pretrained})')
        except:
            raise ValueError('The name of the architecture is not valid!')

    # Modify the fully connected layers after the convolutional part
    # For DenseNet architecture
    if 'densenet' in arch:
        num_in_features = model.classifier.in_features
        model.classifier = nn.Sequential(
            nn.Flatten(),  # Flattening layer
            nn.Linear(num_in_features, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            # nn.Linear(4096, 1024),
            # nn.ReLU(),
            # nn.Linear(1024, 256),
            # nn.ReLU(),
            nn.Linear(4096, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
            nn.Sigmoid()  # Output layer with sigmoid
        )
    else:  # For other architectures if needed (e.g., VGG, AlexNet, ResNet)
        # Similar structure as before for VGG/AlexNet/ResNet
        pass

    return model

# from torchsummary import summary

# # Plotting the structure for a DenseNet169 model
non_trained_densenet169 = build_model(arch='densenet169', pretrained=False).to(device)
# summary(non_trained_densenet169, input_size=(3, 224, 224))

print(non_trained_densenet169)


densenet169_model = build_model(arch='densenet169', pretrained=True)
criterion = torch.nn.CrossEntropyLoss()
learning_rate = 0.01
optimizer = torch.optim.SGD(densenet169_model.parameters(), lr=learning_rate, weight_decay=1e-4)
num_epochs = 30
densenet169_model = train_classifier(model=densenet169_model,
                                     model_name='densenet169_model',
                                     train_loader=train_dl,
                                     valid_loader=test_dl,
                                     criterion=criterion,
                                     optimizer=optimizer,
                                     num_epochs=num_epochs,
                                     plot_curves=True)


In [ ]:
# Import necessary libraries
import torchvision.models as models
import torch.nn as nn

def build_model(arch='vgg16', pretrained=False):
    """
    Build a CNN model using PyTorch with custom dense layers.
    Parameters:
        - arch (str): Model architecture to use from torchvision.models.
        - pretrained (bool): Whether to use pretrained weights.
    """

    # Load specified architecture from torchvision models
    if 'vgg' not in arch and 'alexnet' not in arch and 'resnet' not in arch:
        raise ValueError('Only resnet, vgg, or alexnet models can be loaded!')
    else:
        try:
            model = eval(f'models.{arch}(pretrained={pretrained})')
        except:
            raise ValueError('The name of the architecture is not valid!')

    # Modify the fully connected layers after the convolutional part
    # For VGG and AlexNet architectures
    if 'vgg' in arch or 'alexnet' in arch:
        # Remove original classifier layers and add custom layers
        model.classifier = nn.Sequential(
            nn.Flatten(),  # Flatten the output from the CNN layers
            nn.Linear(25088, 4096),  # Adjust input size if necessary
            nn.ReLU(),
            nn.Linear(4096, 1024),
            nn.ReLU(),
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
            nn.Sigmoid()  # Output layer with softmax
        )
    else:  # For ResNet architectures
        num_in_features = model.fc.in_features
        # Replace fully connected layers for ResNet
        model.fc = nn.Sequential(
            nn.Flatten(),  # Flattening layer
            nn.Linear(num_in_features, 4096),
            nn.ReLU(),
            nn.Linear(4096, 1024),
            nn.ReLU(),
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
            nn.Sigmoid()  # Output layer with softmax
        )

    return model


In [ ]:
from torchsummary import summary

# Plotting the structure for a Vgg16 model
non_trained_vgg16_2 = build_model(arch='vgg16', pretrained=False).to(device)
summary(non_trained_vgg16_2, input_size=(3, 224, 224))

In [ ]:
vgg19_model_2 = build_model(arch='vgg19', pretrained=True)
criterion = torch.nn.CrossEntropyLoss()
learning_rate = 0.001
optimizer = torch.optim.SGD(vgg19_model_2.parameters(), lr=learning_rate)
num_epochs = 30
vgg19_model_2 = train_classifier(model=vgg19_model_2,
                                 model_name='vgg19_model_2',
                                 train_loader=train_dl,
                                 valid_loader=test_dl,
                                 criterion=criterion,
                                 optimizer=optimizer,
                                 num_epochs=num_epochs,
                                 plot_curves=True)

# 7. **Detection**

In [ ]:
# Simple Glaucoma Detection Tool
# Just upload an image and get the glaucoma diagnosis

import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from PIL import Image
from google.colab import drive, files
import seaborn as sns
from IPython.display import display, HTML


# Set model path
MODEL_PATH = '/content/vgg19_model_1_Epoch11_ACC=0.7619047619047619.pth'

# Check if CUDA is available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define image transformations (should match what was used during training)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load the model
def load_model():
    print("Loading model...")
    # Load VGG19 model
    model = torchvision.models.vgg19(pretrained=False)

    # Modify the final fully connected layer for binary classification
    num_features = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(num_features, 2)  # Binary classification: glaucoma/no glaucoma

    # Load the saved model weights
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model = model.to(device)
    model.eval()

    print("Model loaded successfully!")
    return model

# Function to analyze a single image
def analyze_image(model, image):
    # Preprocess the image
    input_tensor = transform(image).unsqueeze(0).to(device)

    # Make prediction
    with torch.no_grad():
        output = model(input_tensor)
        probabilities = torch.softmax(output, dim=1)
        _, predicted = torch.max(output.data, 1)

    # Get results
    class_names = ['Normal', 'Glaucoma']
    prediction = class_names[predicted.item()]
    glaucoma_prob = probabilities[0][1].item()
    normal_prob = probabilities[0][0].item()

    return prediction, glaucoma_prob, normal_prob

# Function to visualize the results
def visualize_results(image, prediction, glaucoma_prob, normal_prob):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))

    # Plot the image with prediction
    ax1.imshow(image)
    ax1.set_title(f"Diagnosis: {prediction}", fontsize=16)
    ax1.axis('off')

    # Create a pie chart showing probabilities
    ax2.pie([glaucoma_prob, normal_prob],
            labels=['Glaucoma', 'Normal'],
            autopct='%1.1f%%',
            startangle=90,
            colors=['#FF9999', '#66B2FF'])
    ax2.axis('equal')
    ax2.set_title('Probability Distribution', fontsize=16)

    plt.tight_layout()
    plt.show()

    # Create a styled HTML display for the results
    risk_level = "High" if glaucoma_prob > 0.7 else "Medium" if glaucoma_prob > 0.3 else "Low"

    html = f"""
    <div style="background-color:#f8f9fa; padding:20px; border-radius:10px; margin:10px 0;">
        <h2 style="color:#333; text-align:center;">Glaucoma Assessment Results</h2>
        <hr style="border:1px solid #ddd;">
        <table style="width:100%; border-collapse:collapse; margin-top:20px;">
            <tr>
                <td style="padding:10px; font-weight:bold; width:40%; color:#000000;">Primary Diagnosis:</td>
                <td style="padding:10px; color:{'#cc0000' if prediction == 'Glaucoma' else '#006600'};">{prediction}</td>
            </tr>
            <tr style="background-color:#f0f0f0;">
                <td style="padding:10px; font-weight:bold; color:#000000;">Glaucoma Probability:</td>
                <td style="padding:10px; color:#000000;">{glaucoma_prob:.1%}</td>
            </tr>
            <tr>
                <td style="padding:10px; font-weight:bold; color:#000000;">Normal Probability:</td>
                <td style="padding:10px; color:#000000;">{normal_prob:.1%}</td>
            </tr>
            <tr style="background-color:#f0f0f0;">
                <td style="padding:10px; font-weight:bold; color:#000000;">Risk Level:</td>
                <td style="padding:10px; color:{'#cc0000' if risk_level == 'High' else '#ff9900' if risk_level == 'Medium' else '#006600'};">{risk_level}</td>
            </tr>
        </table>
        <div style="margin-top:20px; font-size:0.9em; color:#666;">
            <p><strong>Note:</strong> This assessment is based on image analysis using a pre-trained VGG19 model.
            Please consult with an ophthalmologist for proper clinical diagnosis.</p>
        </div>
    </div>
    """

    display(HTML(html))

# Main function to run the application
def run_glaucoma_detector():
    model = load_model()

    print("\n📤 Upload a fundus image to check for glaucoma:")
    uploaded = files.upload()

    for filename in uploaded.keys():
        print(f"\nAnalyzing image: {filename}...")

        # Open and process the image
        image = Image.open(filename).convert('RGB')

        # Get prediction
        prediction, glaucoma_prob, normal_prob = analyze_image(model, image)

        # Display results
        visualize_results(image, prediction, glaucoma_prob, normal_prob)

        print(f"\nDiagnosis complete for {filename}!")
        print(f"Prediction: {prediction}")
        print(f"Glaucoma Probability: {glaucoma_prob:.1%}")
        print(f"Normal Probability: {normal_prob:.1%}")

        print("\nUpload another image or run the cell again to analyze more images.")

# Run the application
print("Glaucoma Detection Tool")
print("------------------------")
run_glaucoma_detector()